In [ ]:
!hostname

In [ ]:
import scanpy as sc
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
# import gseapy as gp
# import decoupler

In [ ]:
import os
os.chdir('/projects/bgdb/asachan/bioinfo_analysis/human_SKM_multimodal_analysis/py_scripts')  # directory containing utils.py
import sys
import logging
import warnings

export_dir = "/projects/bgdb/asachan/datasets/SKM_ageing_human/atac_objects/atac_fiber"

In [ ]:
adata_infile_fiber = '/projects/bgdb/asachan/datasets/SKM_ageing_human/rna_objects/Myofiber_scsn_RNA.h5ad'
adata_female_rna = '/projects/bgdb/asachan/datasets/SKM_ageing_human/atac_objects/atac_fiber/rna_female_type2_noP21.h5ad'
adata_female_atac = '/projects/bgdb/asachan/datasets/SKM_ageing_human/atac_objects/atac_fiber/atac_female_type2_noP21.h5ad'
out_tmp = '/projects/bgdb/asachan/datasets/SKM_ageing_human/tmp'

In [ ]:
rna_adata = sc.read_h5ad(adata_female_rna)

In [ ]:
adata_female_downsampled.obs['age_pop'] = pd.Categorical(adata_female_downsampled.obs['age_pop'], categories=["old_pop", "young_pop"], ordered=True)

In [ ]:
bdata = adata_female_downsampled[adata_female_downsampled.obs.Annotation == "Type I"].copy()
bdata

In [ ]:
sc.tl.rank_genes_groups(bdata,
                        groupby='age_pop',
                        use_raw=False,
                        method='wilcoxon',
                        groups=["old_pop"],
                        reference='young_pop')

In [ ]:
sc.pl.rank_genes_groups(bdata, n_genes=25, sharey=False)

In [ ]:
# get deg result
result = bdata.uns['rank_genes_groups']
groups = result['names'].dtype.names
degs = pd.DataFrame(
    {group + '_' + key: result[key][group]
    for group in groups for key in ['names','scores', 'pvals','pvals_adj','logfoldchanges']})
degs.head()

In [ ]:
# subset up or down regulated genes
degs_sig = degs[degs.old_pop_pvals_adj < 0.01]
degs_up = degs_sig[degs_sig.old_pop_logfoldchanges > 0]
degs_dw = degs_sig[degs_sig.old_pop_logfoldchanges < 0]

In [ ]:
degs_up.shape

In [ ]:
degs_dw.shape

In [ ]:
enr_up = gp.enrichr(degs_up.old_pop_names,
                    gene_sets='GO_Biological_Process_2025',
                    outdir=None)

In [ ]:
enr_up.res2d

In [ ]:
# dotplot
enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
gp.dotplot(enr_up.res2d, figsize=(3,5), title="Up", cmap = plt.cm.autumn_r)
plt.show()

In [ ]:
enr_dw = gp.enrichr(degs_dw.old_pop_names,
                    gene_sets='GO_Biological_Process_2025',
                    outdir=None)

In [ ]:
enr_dw.res2d

In [ ]:
enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
gp.dotplot(enr_dw.res2d,
           figsize=(3,5),
           title="Down",
           cmap = plt.cm.winter_r,
           size=5)
plt.show()